# 02 - Verificacion de Eventos en Kafka

Este notebook verifica que los eventos del producer esten llegando correctamente a Kafka.

## 1. Instalacion de kafka-python

In [1]:
import sys
!{sys.executable} -m pip install kafka-python-ng pandas python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.8/232.8 kB 4.9 MB/s eta 0:00:00a 0:00:01


## 2. Consumo de mensajes de Kafka

In [2]:
from kafka import KafkaConsumer
import json
import time

KAFKA_BROKER = "kafka:29092"
TOPIC = "iot.air_quality.streaming"
MAX_MENSAJES = 20
TIEMPO_ESPERA = 30

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset="latest",
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    consumer_timeout_ms=1000
)

print(f"Consumiendo mensajes del topico: {TOPIC}")
print(f"Esperando hasta {TIEMPO_ESPERA}s o {MAX_MENSAJES} mensajes...\n")

messages = []
start = time.time()
try:
    while time.time() - start < TIEMPO_ESPERA and len(messages) < MAX_MENSAJES:
        msg_pack = consumer.poll(timeout_ms=1000)
        for tp, records in msg_pack.items():
            for record in records:
                messages.append(record.value)
                print(f"[{len(messages)}] Estacion: {record.value.get('estacion')} | Temp: {record.value.get('temperatura')} | IAQ: {record.value.get('iaq')}")
except KeyboardInterrupt:
    print("\n[Consumo interrumpido por el usuario]")

print(f"\nTotal mensajes recibidos: {len(messages)}")
consumer.close()

Consumiendo mensajes del topico: iot.air_quality.streaming
Esperando hasta 30s o 20 mensajes...

[1] Estacion: ESP32_01 | Temp: 10.23 | IAQ: 0
[2] Estacion: ESP32_05 | Temp: 10.21 | IAQ: 0
[3] Estacion: ESP32_02 | Temp: 10.49 | IAQ: 0
[4] Estacion: ESP32_04 | Temp: 10.03 | IAQ: 15
[5] Estacion: ESP32_03 | Temp: 9.6 | IAQ: 0
[6] Estacion: ESP32_02 | Temp: 10.45 | IAQ: 0
[7] Estacion: ESP32_03 | Temp: 10.62 | IAQ: 1
[8] Estacion: ESP32_04 | Temp: 10.21 | IAQ: 23
[9] Estacion: ESP32_01 | Temp: 10.23 | IAQ: 0
[10] Estacion: ESP32_05 | Temp: 10.27 | IAQ: 0
[11] Estacion: ESP32_03 | Temp: 10.36 | IAQ: 3
[12] Estacion: ESP32_04 | Temp: 10.29 | IAQ: 20
[13] Estacion: ESP32_01 | Temp: 10.23 | IAQ: 0
[14] Estacion: ESP32_02 | Temp: 10.15 | IAQ: 0
[15] Estacion: ESP32_05 | Temp: 10.29 | IAQ: 3
[16] Estacion: ESP32_01 | Temp: 10.23 | IAQ: 0
[17] Estacion: ESP32_04 | Temp: 10.23 | IAQ: 23
[18] Estacion: ESP32_02 | Temp: 10.03 | IAQ: 3
[19] Estacion: ESP32_03 | Temp: 11.41 | IAQ: 0
[20] Estacion: ES

## 3. Analisis de mensajes recibidos

In [3]:
import pandas as pd

if messages:
    df = pd.DataFrame(messages)
    print("\nEstructura del evento:")
    print(df.columns.tolist())
    print("\nPrimer evento completo:")
    print(df.iloc[0].to_dict())
    print("\nEstaciones detectadas:")
    print(df["estacion"].value_counts())
else:
    print("No se recibieron mensajes. Verifica que el producer este corriendo.")


Estructura del evento:
['estacion', 'temperatura', 'humedad', 'presion', 'altura', 'gas', 'iaq', 'eco2', 'VOC', 'calidad_aire', 'created_at', 'event_timestamp', 'delayed']

Primer evento completo:
{'estacion': 'ESP32_01', 'temperatura': 10.23, 'humedad': 36.36, 'presion': 646.23, 'altura': 3622.35, 'gas': 28445, 'iaq': 0, 'eco2': 408, 'VOC': 0.528, 'calidad_aire': 'BUENA', 'created_at': '2026-06-28T02:12:54.656053', 'event_timestamp': 1782612774656, 'delayed': nan}

Estaciones detectadas:
estacion
ESP32_01    4
ESP32_05    4
ESP32_02    4
ESP32_04    4
ESP32_03    4
Name: count, dtype: int64


## 4. Verificacion de eventos retrasados (ESP32_05)

In [4]:
if messages:
    delayed = [m for m in messages if m.get("delayed", False)]
    print(f"Eventos retrasados detectados: {len(delayed)}")
    if delayed:
        print("\nEjemplo de evento retrasado:")
        print(f"  created_at: {delayed[0].get('created_at')}")
        print(f"  estacion: {delayed[0].get('estacion')}")
        print(f"  delayed: {delayed[0].get('delayed')}")

Eventos retrasados detectados: 4

Ejemplo de evento retrasado:
  created_at: 2026-06-28T02:12:49.656452
  estacion: ESP32_05
  delayed: True


## 5. Verificacion de latencia

In [5]:
from datetime import datetime

if messages:
    latencias = []
    now = datetime.utcnow().timestamp() * 1000
    for m in messages:
        if m.get("event_timestamp"):
            lat = now - m["event_timestamp"]
            latencias.append(lat)
    
    if latencias:
        print(f"Latencia promedio: {sum(latencias)/len(latencias):.0f} ms")
        print(f"Latencia minima: {min(latencias):.0f} ms")
        print(f"Latencia maxima: {max(latencias):.0f} ms")

Latencia promedio: 16312 ms
Latencia minima: 308 ms
Latencia maxima: 35317 ms
